<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson05/02_datasets_and_tokenizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05 · Notebook 2: 🤗 Datasets and Tokenizers

In notebook 1 our Shakespeare GPT ended up on the Hub, but with a home-made `chars.txt` as its vocabulary. Now we meet two more Hugging Face libraries:

- `datasets`: load a dataset from the Hub with one line, look at it, filter it, and share your own
- `tokenizers`: train a real BPE tokenizer in a second (last week our pure-Python version took several seconds for just 30 merges)

At the end we train the Shakespeare GPT once more, now with a proper tokenizer, and upload both to the Hub. Then the model works with `pipeline()`, like any other model on the Hub.

**Contents:** 0. Setup · 1. Loading a dataset · 2. Looking inside · 3. `filter`, `map` and friends · 4. Streaming huge datasets · 5. Your own dataset on the Hub · 6. Training a tokenizer · 7. A tokenizer for Czech · 8. Shakespeare GPT with a real tokenizer · 9. Upload, and `pipeline` · 10. Summary

**Before you start:** *Runtime → Change runtime type → T4 GPU* (for section 8). For sections 5 and 9 you need the `HF_TOKEN` secret from notebook 1.

---
## 0. Setup

In [ ]:
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import datasets
import tokenizers

print("datasets", datasets.__version__, "  tokenizers", tokenizers.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# @title Helper function (just run this cell)
import html
from IPython.display import HTML, display

COLORS = ["#fde68a", "#bfdbfe", "#bbf7d0", "#fbcfe8", "#ddd6fe", "#fed7aa"]

def show_tokens(pieces):
    # Show text pieces in alternating colors, like the Tiktokenizer app (as in lesson 04).
    spans = []
    for i, piece in enumerate(pieces):
        piece = html.escape(piece).replace("\n", "↵\n")
        spans.append(f'<span style="background:{COLORS[i % len(COLORS)]}; color:black; border-radius:3px">{piece}</span>')
    display(HTML(f'<pre style="font-size:15px; line-height:1.8; white-space:pre-wrap; margin:0">{"".join(spans)}</pre>'
                 f'<div style="font-size:12px; margin-bottom:8px">{len(pieces)} tokens</div>'))

---
## 1. Loading a dataset

The Hub hosts hundreds of thousands of datasets. Our example: 30,000 Czech user reviews from [ČSFD](https://www.csfd.cz), the Czech movie database. Open its page first: [huggingface.co/datasets/fewshot-goes-multilingual/cs_csfd-movie-reviews](https://huggingface.co/datasets/fewshot-goes-multilingual/cs_csfd-movie-reviews). The **dataset viewer** there lets you scroll through the data before you download anything.

`load_dataset` downloads it (the first time) and loads it:

In [ ]:
from datasets import load_dataset

reviews = load_dataset("fewshot-goes-multilingual/cs_csfd-movie-reviews")
reviews

A `DatasetDict`: three **splits** (train, validation, test), and in each a `Dataset`, a table with named columns (*features*) and rows.

---
## 2. Looking inside

One row is a Python dictionary:

In [ ]:
train = reviews["train"]
train[0]

A slice gives a dictionary of **lists**, one list per column. So `train[:3]["comment"]` are the first three reviews, and `train["rating_int"]` is the whole column:

In [ ]:
for comment in train[:3]["comment"]:
    print(comment.strip()[:200])
print(train["rating_int"][:20])

Every column has a **type**, so the data is stored and loaded the same way on every computer:

In [ ]:
train.features

**Look inside: where is the data?** Not in memory! `datasets` stores the table in a file in the **Apache Arrow** format and reads only the rows you ask for. That is why it can open datasets much bigger than your RAM.

In [ ]:
print(train.cache_files)
!ls -lh ~/.cache/huggingface/datasets

For exploring, `pandas` is often handier. `to_pandas()` loads the whole table into a `DataFrame`:

In [ ]:
df = train.to_pandas()
print(df["item_title"].value_counts().head(10))

df["rating_int"].value_counts().sort_index().plot.bar(title="ratings (0-5 stars)")
plt.show()

The ratings are **balanced**: the authors of the dataset chose about the same number of reviews for every number of stars. On the real ČSFD there are many more good ratings than bad ones.

---
## 3. `filter`, `map` and friends

`filter` keeps the rows for which a function returns `True`:

In [ ]:
avatar = train.filter(lambda row: row["item_title"] == "Avatar")
print(len(avatar), "reviews of Avatar")
for row in avatar.select(range(3)):           # select: rows by their position
    print(row["rating_str"], row["comment"].strip()[:150])

`map` runs a function on every row. The function returns a dictionary, and its keys become **new columns**:

In [ ]:
def count_words(row):
    return {"n_words": len(row["comment"].split())}

train = train.map(count_words)
print(train)
print("average review length:", sum(train["n_words"]) / len(train), "words")

A few more useful methods:

In [ ]:
sample = train.shuffle(seed=42).select(range(1000))          # 1,000 random rows
parts = sample.train_test_split(test_size=0.2, seed=42)      # a new random split
print(parts)
print(sample.sort("n_words")[-1]["n_words"], "words in the longest review of the sample")

**Try it:**
- How many reviews does your favourite movie have, and what is its average rating?
- Do longer reviews give better or worse ratings? (Hint: `df.groupby("rating_int")["comment"].apply(lambda s: s.str.len().mean())`.)
- Which director has the best average rating? (`item_directors` is a list, `df.explode("item_directors")` makes one row per director.)

---
## 4. Streaming huge datasets

The Czech Wikipedia has 534,044 articles, about 1 GB to download (5.6 GB as an Arrow table). With `streaming=True` nothing is downloaded up front: the rows arrive one by one, while you read them.

In [ ]:
wiki = load_dataset("wikimedia/wikipedia", "20231101.cs", split="train", streaming=True)

for article in wiki.take(3):           # only the first 3 articles
    print(f"{article['title']}: {len(article['text']):,} characters")
    print("   ", article["text"][:200].replace("\n", " "))

This is how big models are trained: FineWeb, a typical LLM training set, has 15 trillion tokens, far more than fits on one disk.

---
## 5. Your own dataset on the Hub

`load_dataset` can also read local files: CSV, JSON, Parquet, or plain text. Our tiny Shakespeare, with `sample_by="paragraph"`, becomes one row per **speech** (the text between empty lines):

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

shakespeare = load_dataset("text", data_files="input.txt", sample_by="paragraph")
print(shakespeare)
print(shakespeare["train"][0]["text"])

Now let's share it. First we log in, as in notebook 1 (with the `HF_TOKEN` secret it happens automatically):

In [ ]:
from huggingface_hub import login, whoami

try:
    user = whoami()["name"]
except Exception:            # no HF_TOKEN secret: paste a token with "Write" access
    login()
    user = whoami()["name"]
print("logged in as", user)

In [ ]:
shakespeare.push_to_hub(f"{user}/tiny-shakespeare")
print(f"https://huggingface.co/datasets/{user}/tiny-shakespeare")

Open the link: your dataset has its own page with the dataset viewer. Anyone can now load it with `load_dataset("your-username/tiny-shakespeare")`. (Datasets are uploaded as Parquet files, a compressed table format. Look at *Files and versions*.)

---
## 6. Training a tokenizer

Last week we trained Byte Pair Encoding in pure Python. 30 merges took several seconds, and GPT-2 has 50,000 of them. The `tokenizers` library does the same in Rust, a fast compiled language.

A tokenizer has several parts, and we choose each of them:
- **pre-tokenizer**: turns text into bytes and cuts it into word-like chunks (`ByteLevel` does exactly what GPT-2 does: the regex from last week, and a space at the *start* of a word)
- **model**: the BPE merges themselves
- **decoder**: turns tokens back into text

The **trainer** says how big the vocabulary should be. We ask for 1,000 tokens: 256 bytes, one special token `<|endoftext|>`, and 743 merges.

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

def new_bpe_tokenizer(vocab_size):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<|endoftext|>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # all 256 bytes are always in the vocabulary
    )
    return tokenizer, trainer

tokenizer, trainer = new_bpe_tokenizer(vocab_size=1000)
t0 = time.time()
tokenizer.train(["input.txt"], trainer)
print(f"trained in {time.time() - t0:.2f} s, vocabulary of {tokenizer.get_vocab_size()} tokens")

743 merges on 1 MB of text in a fraction of a second. Let's use it:

In [ ]:
enc = tokenizer.encode("To be, or not to be: that is the question.")
print(enc.ids)
print(enc.tokens)

What is `Ġ`? GPT-2's byte-level trick: every byte gets a printable character, so that the vocabulary can be stored as text, and the space became `Ġ`. To see real text, we **decode** every token on its own:

In [ ]:
def pieces(tokenizer, text):
    # The text cut into tokens, each token decoded back to readable text.
    return [tokenizer.decode([i]) for i in tokenizer.encode(text).ids]

show_tokens(pieces(tokenizer, "To be, or not to be: that is the question.\nROMEO:\nO Romeo, Romeo, wherefore art thou Romeo?"))

**Look inside the vocabulary.** The first merges are the most frequent letter pairs, as last week. The longest tokens are... the names of Shakespeare's characters, which appear before every speech:

In [ ]:
vocab = tokenizer.get_vocab()                       # token -> id
by_id = sorted(vocab, key=vocab.get)
print("first merges:  ", [tokenizer.decode([vocab[t]]) for t in by_id[257:277]])
print("longest tokens:", [tokenizer.decode([vocab[t]]) for t in sorted(vocab, key=len)[-12:-1]])

How many tokens is all of Shakespeare? Let's compare our tokenizer with GPT-2's, which has 50,257 tokens. `AutoTokenizer` downloads a model's tokenizer from the Hub:

In [ ]:
from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

with open("input.txt", encoding="utf-8") as f:
    text = f.read()
print(f"characters:                    {len(text):>9,}")
print(f"our tokenizer (1,000 tokens):  {len(tokenizer.encode(text).ids):>9,}")
print(f"GPT-2 tokenizer (50,257):      {len(gpt2_tokenizer(text)['input_ids']):>9,}")

*(You can ignore the warning that the sequence is longer than 1,024: we only count tokens, we do not give them to GPT-2.)*

A bigger vocabulary means fewer tokens, but also a bigger embedding table. For a tiny model trained on 1 MB of text, 1,000 tokens is a good compromise.

---
## 7. A tokenizer for Czech

Last week we saw that GPT-2 needs many tokens for Czech: its merges were learned on English. Our Shakespeare tokenizer is even worse. So let's train a tokenizer on **Czech** text, the 25,000 ČSFD reviews from section 1.

The reviews are not in a file, so we use `train_from_iterator` and feed the texts in batches of 1,000:

In [ ]:
def review_batches():
    for i in range(0, len(train), 1000):
        yield train[i:i + 1000]["comment"]

czech_tokenizer, trainer = new_bpe_tokenizer(vocab_size=8000)
czech_tokenizer.train_from_iterator(review_batches(), trainer, length=len(train))

sentence = "Praha je hlavní město České republiky. Tenhle film byl naprostá nuda, ale herci hráli skvěle."
print("GPT-2:"); show_tokens([gpt2_tokenizer.decode([i]) for i in gpt2_tokenizer(sentence)["input_ids"]])
print("Shakespeare:"); show_tokens(pieces(tokenizer, sentence))
print("ČSFD reviews:"); show_tokens(pieces(czech_tokenizer, sentence))

Whole Czech words, often with their endings, became single tokens. *(Some tokens are only half of a letter, shown as `�`: letters like `ě` take 2 bytes, and a token may contain just one of them.)*

**Try it:** Which are the longest tokens of the Czech tokenizer? (Copy the "longest tokens" line from section 6.) And how many tokens does `czech_tokenizer` need for a piece of Shakespeare?

---
## 8. Shakespeare GPT with a real tokenizer

To use our tokenizer with `transformers`, we wrap it in `PreTrainedTokenizerFast`. Then it behaves like any tokenizer from the Hub, e.g. like `gpt2_tokenizer` above: you call it on a text, and it can `decode`, `save_pretrained` and `push_to_hub`:

In [ ]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer, eos_token="<|endoftext|>")

print(hf_tokenizer("To be, or not to be"))
print(hf_tokenizer.decode(hf_tokenizer("To be, or")["input_ids"]))

Now the same steps as in notebook 1, only with tokens instead of characters:

In [ ]:
data = torch.tensor(hf_tokenizer(text)["input_ids"])
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
chars_per_token = len(text) / len(data)
print(f"{len(data):,} tokens, on average {chars_per_token:.2f} characters per token")

context = 128          # now 128 tokens, about 300 characters
batch_size = 64

def get_batch(split):
    d = train_data if split == "train" else val_data
    starts = torch.randint(len(d) - context, (batch_size,))
    return torch.stack([d[s:s + context] for s in starts]).to(device)

The model: the same as in notebook 1, except for the vocabulary size. We also switch **dropout** on. Without it, this model starts to memorize the training text after about 1,500 steps (see the *Try it* below).

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=len(hf_tokenizer),                     # 1,000 tokens
    n_positions=context, n_embd=128, n_layer=4, n_head=4,
    resid_pdrop=0.1, embd_pdrop=0.1, attn_pdrop=0.1,  # dropout on
    bos_token_id=hf_tokenizer.eos_token_id,
    eos_token_id=hf_tokenizer.eos_token_id,           # our <|endoftext|> token now exists
)
torch.manual_seed(1337)
model = GPT2LMHeadModel(config).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

The embedding table grew from 65 × 128 to 1,000 × 128, so we have about 120,000 more parameters than in notebook 1.

The training loop is copied from notebook 1. One new detail: `model.eval()` switches dropout off while we measure the loss, and `model.train()` switches it back on.

In [ ]:
@torch.no_grad()
def estimate_loss(model, n_batches=20):
    model.eval()
    result = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(n_batches):
            x = get_batch(split)
            losses.append(model(x, labels=x).loss.item())
        result[split] = sum(losses) / len(losses)
    model.train()
    return result

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
steps = 2000 if device == "cuda" else 300
history = []
t0 = time.time()

for step in range(1, steps + 1):
    x = get_batch("train")
    loss = model(x, labels=x).loss       # 1. FORWARD + 2. LOSS
    optimizer.zero_grad()                # 5. RESET
    loss.backward()                      # 3. BACKWARD
    optimizer.step()                     # 4. UPDATE

    if step % 250 == 0 or step == steps:
        losses = estimate_loss(model)
        history.append((step, losses["train"], losses["val"]))
        print(f"step {step:5}:  train loss {losses['train']:.3f}   validation loss {losses['val']:.3f}   ({time.time() - t0:.0f} s)")

In [ ]:
done, train_losses, val_losses = zip(*history)
plt.plot(done, train_losses, label="training")
plt.plot(done, val_losses, label="validation")
plt.xlabel("step")
plt.ylabel("loss per token")
plt.legend()
plt.show()

**Look inside: is it better than notebook 1?** The losses are much higher than in notebook 1, but that is not a fair comparison. The loss is measured **per token**, and guessing the next token (about 2.4 characters) is harder than guessing the next character. To compare, we divide by the number of characters per token:

In [ ]:
print(f"validation loss per token:     {history[-1][2]:.3f}")
print(f"validation loss per character: {history[-1][2] / chars_per_token:.3f}    (compare with the end of training in notebook 1)")

*(This is only roughly right, because the validation text has slightly fewer characters per token than the average.)*

Let's generate some text. `generate` now gets the ids from our tokenizer, and the prompt plus new tokens must fit into 128 tokens:

In [ ]:
model.eval()
ids = hf_tokenizer("ROMEO:\n", return_tensors="pt")["input_ids"].to(device)
torch.manual_seed(42)
out = model.generate(ids, max_new_tokens=100, do_sample=True, temperature=0.8)
print(hf_tokenizer.decode(out[0]))

**Try it:** Switch the dropout off (`0.0`) and train for 3,000 steps. Watch the validation loss: after about 1,500 steps it starts to **grow**, while the training loss keeps falling. The model **memorizes** the training text instead of learning Shakespeare's style: *overfitting*. With tokens, the 1 MB of text is only about 460,000 tokens, so the model sees each one many more times than in notebook 1.

---
## 9. Upload, and `pipeline`

We upload the model **and** the tokenizer into the same repository:

In [ ]:
repo_id = f"{user}/shakespeare-gpt-bpe"
model.push_to_hub(repo_id)
hf_tokenizer.push_to_hub(repo_id)
print(f"https://huggingface.co/{repo_id}")

Look at *Files and versions*: next to `config.json` and `model.safetensors` there is now `tokenizer.json`, the whole tokenizer (all parts from section 6 and the 743 merges) in one text file.

And now the reward: any model on the Hub that has a tokenizer works with `pipeline`, which takes care of tokenizing, generating and decoding. Try your neighbour's model too:

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=repo_id, device=device)
torch.manual_seed(0)
result = generator("JULIET:\n", max_new_tokens=100, do_sample=True, temperature=0.8)
print(result[0]["generated_text"])

---
## 10. Summary

| step | code |
|---|---|
| loading a dataset | `load_dataset("user/name")`, with `streaming=True` for huge ones |
| looking at it | `ds[0]`, `ds[:5]["column"]`, `ds.features`, `ds.to_pandas()` |
| changing it | `filter`, `map`, `select`, `shuffle`, `sort`, `train_test_split` |
| your own data | `load_dataset("text" / "csv" / "json", data_files=...)`, `push_to_hub` |
| training a tokenizer | `Tokenizer(models.BPE())` + pre-tokenizer + decoder, `BpeTrainer`, `train` / `train_from_iterator` |
| using it in `transformers` | `PreTrainedTokenizerFast(tokenizer_object=...)`, `push_to_hub` |
| the easy way | `pipeline("text-generation", model="user/name")` |

**New words:** dataset split, features, Apache Arrow, Parquet, streaming, pre-tokenizer, special token, vocabulary size, overfitting, dropout, pipeline.

### Exercises
1. Train the Shakespeare GPT with a vocabulary of 300 and of 5,000 tokens. Compare the validation loss **per character**. Which vocabulary size is the best?
2. Train a GPT on the ČSFD reviews (all `comment`s joined with `"<|endoftext|>"`), with `czech_tokenizer`. Can it write a movie review?
3. Upload a dataset of your own (e.g. a CSV file from an earlier project) and look at it in the dataset viewer.

### Further reading
- [Datasets documentation](https://huggingface.co/docs/datasets), [Tokenizers documentation](https://huggingface.co/docs/tokenizers)
- Hugging Face LLM course: [chapter 5, the Datasets library](https://huggingface.co/learn/llm-course/chapter5/1) and [chapter 6, the Tokenizers library](https://huggingface.co/learn/llm-course/chapter6/1)
- [FineWeb](https://huggingface.co/spaces/HuggingFaceFW/blogpost-fineweb-v1): how a 15-trillion-token training set is made